# RRUFF Preprocessing
Returns a PyG `Data` object with:
- `x` : node feature matrix (CLIP image emb + Raman emb)
- `edge_index` : graph edges (3 types concatenated)
- `edge_type` : which edge type each edge is
- `y` : mineral class labels (integer)
- `mineral_names` : list of mineral names (index = node id)

In [ ]:
import torch
print(torch.__version__)  # should print 2.1.0+cu128 or 2.10.0+cu128

2.10.0+cu128


In [ ]:
# ── Install deps (fast — hardcoded versions) ──────────────────────────────────
!pip install torch-geometric -q
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.10.0+cu128.html -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 123.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 140.2 MB/s eta 0:00:00


In [ ]:
!pip install open-clip-torch scipy scikit-learn tqdm -q
print("Done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.2 MB/s eta 0:00:00
Done.


In [ ]:
try:
    import torch_geometric, torch_scatter, open_clip
    print("All packages already installed — skipping.")
except ImportError:
    # run the install block
    ...

All packages already installed — skipping.


In [ ]:
!unzip /content/rruff_good_images.zip -d /content/rruff_good_images

Archive:  /content/rruff_good_images.zip
  inflating: /content/rruff_good_images/Abelsonite__R070007__Sample__Photo__9ef62df30c9bc7b3bed4c7c8de0d.jpeg  
  inflating: /content/rruff_good_images/Acanthite__R080016__Sample__Photo__faede17ec56face9f92b783aaacf.jpeg  
  inflating: /content/rruff_good_images/Actinolite__R060041__Sample__Photo__4c5f6567d5b3f03adfae0edf7a36.jpeg  
  inflating: /content/rruff_good_images/Actinolite__R060045__Sample__Photo__9278f62cc13e7d2d662ad695e485.jpeg  
  inflating: /content/rruff_good_images/Adamite__R060593__Sample__Photo__c76ef6978dd367d82b8413c485f6.jpeg  
  inflating: /content/rruff_good_images/Aegirine__R050074__Sample__Photo__05746b3c8883a3ba22b7096ff15b.jpeg  
  inflating: /content/rruff_good_images/Aegirine__R070253__Sample__Photo__a8acd6781c106fa149b3c05e1d56.jpeg  
  inflating: /content/rruff_good_images/Aegirine__R120144__Sample__Photo__b91fc481db18ec96ac31fc374ec9.jpeg  
  inflating: /content/rruff_good_images/Aegirine__R120167__Sample__Photo_

In [ ]:
!unzip /content/LR-Raman.zip -d /content/LR-Raman

Streaming output truncated to the last 5000 lines.
  inflating: /content/LR-Raman/Lefontite__R140428__Broad_Scan__532__0__unoriented__Raman_Data_Processed__427748dc880dfe82b3bf6ffdfbe7.txt  
  inflating: /content/LR-Raman/Lefontite__R140428__Broad_Scan__780__0__unoriented__Raman_Data_Processed__b00e601d702c8377ca0234505aef.txt  
  inflating: /content/LR-Raman/Lefontite__R140539__Broad_Scan__532__0__unoriented__Raman_Data_Processed__ac49a3f17c6132e72c5562ddbb9a.txt  
  inflating: /content/LR-Raman/Lefontite__R140539__Broad_Scan__780__0__unoriented__Raman_Data_Processed__ffed34377807ccde90e250746028.txt  
  inflating: /content/LR-Raman/Legrandite__R040151__Broad_Scan__532__0__unoriented__Raman_Data_Raw__dfda785b8e19da419602774c3e98.txt  
  inflating: /content/LR-Raman/Legrandite__R040151__Broad_Scan__785__0__unoriented__Raman_Data_Raw__0abda95df16c46811745ee5e001b.txt  
  inflating: /content/LR-Raman/Leifite__R060219__Broad_Scan__532__0__unoriented__Raman_Data_Raw__493cad23db8b6f9baf39a1

In [ ]:
import os

#sanity check that local input folders must exist
LOCAL_IMG   = "/content/rruff_good_images"
LOCAL_RAMAN = "/content/LR-Raman"
LOCAL_CSV   = "/content/working_set.csv"

missing = [p for p in [LOCAL_IMG, LOCAL_RAMAN, LOCAL_CSV] if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        f"Missing local inputs — upload these to /content/ before running:\n"
        + "\n".join(f"  {p}" for p in missing))
print("✓ Local input files found.")

from google.colab import drive
drive.mount('/content/drive')

import torch
TORCH = torch.__version__.split('+')[0]
CUDA  = 'cu' + torch.version.cuda.replace('.', '')
print(f"PyTorch: {TORCH}, CUDA: {CUDA}")


✓ Local input files found.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.10.0, CUDA: cu128


In [ ]:
# ── Cell 2: Imports & paths ───────────────────────────────────────────────────
import os, re, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn.functional as F
import open_clip
from sklearn.preprocessing import LabelEncoder
import torch_geometric
from torch_geometric.data import Data

warnings.filterwarnings('ignore')

#--input paths locally uploaded to /content/ on colab--
IMG_DIR   = Path("/content/rruff_good_images")
RAMAN_DIR = Path("/content/LR-Raman")
WS_CSV    = Path("/content/working_set.csv")

#--output paths--
# Local fast storage for intermediate files; Drive for resumable checkpoints
DRIVE_OUT = Path("/content/drive/My Drive/Classes/2026 Spring/advanced ml/rruff_processed")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
OUT_DIR   = Path("/content/rruff_processed")   # fast local scratch
OUT_DIR.mkdir(exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

RRUFF_ID_RE = re.compile(r'(R\d{6})', re.IGNORECASE)
MINERAL_RE  = re.compile(r'^([A-Za-z][^_]*)__')

def parse_filename(fname):
    stem = Path(fname).stem
    id_match   = RRUFF_ID_RE.search(stem)
    name_match = MINERAL_RE.match(stem)
    rruff_id     = id_match.group(1).upper()   if id_match   else None
    mineral_name = name_match.group(1).strip() if name_match else None
    return rruff_id, mineral_name


Device: cuda


In [ ]:
import subprocess, sys, os, json as _json, time, random, shutil
import PIL.ImageFilter
from PIL import Image, ImageDraw
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'google-cloud-vision', 'simple-lama-inpainting'], check=True)
print("✓ Deps installed.")

import cv2
for _name, _val in {'INTER_NEAREST': 0, 'INTER_LINEAR': 1, 'INTER_CUBIC': 2,
                    'INTER_AREA': 3, 'INTER_LANCZOS4': 4}.items():
    if not hasattr(cv2, _name):
        setattr(cv2, _name, _val)

VISION_BATCH_SIZE = 16
VISION_WORKERS    = 4
LAMA_BATCH_SIZE   = 8
MASK_DILATION     = 14
CHECKPOINT_EVERY  = 100
SAVE_WORKERS      = 4
MAX_RETRIES       = 5
RETRY_BASE_S      = 2.0

CLEAN_IMG_LOCAL = Path("/content/rruff_clean_images")
CLEAN_IMG_DRIVE = DRIVE_OUT / "rruff_clean_images"
VISION_CACHE    = DRIVE_OUT / "vision_cache.json"
INPAINT_LOG     = DRIVE_OUT / "inpaint_done.txt"

CLEAN_IMG_LOCAL.mkdir(exist_ok=True)
CLEAN_IMG_DRIVE.mkdir(parents=True, exist_ok=True)

from google.colab import userdata
from google.cloud import vision as gvision
from google.api_core.client_options import ClientOptions

vision_client = gvision.ImageAnnotatorClient(
    client_options=ClientOptions(api_key=userdata.get('google-vision-api').strip()))
print("✓ Vision API client ready.")

from simple_lama_inpainting import SimpleLama
import torch, numpy as np

lama = SimpleLama()
lama_device = next(lama.model.parameters()).device
print(f"✓ LaMa ready on {lama_device}.")

# google vision api is used to detect text in images (since some images are labeled with their RRUFFID, which would affect training)
vision_cache = {}
if VISION_CACHE.exists():
    with open(VISION_CACHE) as fh:
        vision_cache = _json.load(fh)
    print(f"Loaded Vision cache: {len(vision_cache)} images already processed.")

all_images  = sorted([f for f in IMG_DIR.iterdir()
                       if f.suffix.lower() in ('.jpg', '.jpeg', '.png')])
todo_vision = [f for f in all_images if f.name not in vision_cache]
print(f"\nStage 1 — Vision API: {len(todo_vision)} to annotate "
      f"({len(all_images) - len(todo_vision)} cached).")

def _annotate_batch(batch_paths):
    requests = []
    for p in batch_paths:
        with open(p, 'rb') as fh:
            content = fh.read()
        requests.append(gvision.AnnotateImageRequest(
            image=gvision.Image(content=content),
            features=[gvision.Feature(type_=gvision.Feature.Type.TEXT_DETECTION)],
        ))
    for attempt in range(MAX_RETRIES):
        try:
            response = vision_client.batch_annotate_images(requests=requests)
            results  = []
            for p, r in zip(batch_paths, response.responses):
                boxes = (
                    [[[v.x, v.y] for v in ann.bounding_poly.vertices]
                     for ann in r.text_annotations[1:]]
                    if r.text_annotations else []
                )
                results.append((p.name, boxes))
            return results
        except Exception as e:
            if '429' in str(e) and attempt < MAX_RETRIES - 1:
                time.sleep(RETRY_BASE_S * (2 ** attempt) + random.uniform(0, 1))
            else:
                raise

if todo_vision:
    batches = [todo_vision[i:i + VISION_BATCH_SIZE]
               for i in range(0, len(todo_vision), VISION_BATCH_SIZE)]
    with ThreadPoolExecutor(max_workers=VISION_WORKERS) as pool:
        futures = {pool.submit(_annotate_batch, b): b for b in batches}
        for fut in tqdm(as_completed(futures), total=len(batches),
                        desc='Vision API batches'):
            try:
                for name, boxes in fut.result():
                    vision_cache[name] = boxes
            except Exception as e:
                print(f"  ⚠ Batch failed after retries: {e}")
    with open(VISION_CACHE, 'w') as fh:
        _json.dump(vision_cache, fh)
    print(f"Vision cache saved to Drive ({len(vision_cache)} entries).")

#---LaMa inpainting local only w Drive sync at end---
inpaint_done = set()
if INPAINT_LOG.exists():
    inpaint_done = set(INPAINT_LOG.read_text().splitlines())
    print(f"\nInpainting log: {len(inpaint_done)} images already done.")

def _make_mask(img_size, boxes):
    mask = Image.new('L', img_size, 0)
    if boxes:
        draw = ImageDraw.Draw(mask)
        for pts in boxes:
            draw.polygon([(p[0], p[1]) for p in pts], fill=255)
        mask = mask.filter(PIL.ImageFilter.MaxFilter(size=2 * MASK_DILATION + 1))
    return mask

def _preprocess(img, mask):
    orig = img.size
    W, H = (img.width + 7) // 8 * 8, (img.height + 7) // 8 * 8
    t_img  = torch.from_numpy(np.array(img.resize((W, H), Image.LANCZOS))).float() / 255.0
    t_img  = t_img.permute(2, 0, 1)
    t_mask = (torch.from_numpy(np.array(mask.resize((W, H), Image.NEAREST))).float() / 255.0 > 0.5).float().unsqueeze(0)
    return t_img, t_mask, orig

def _postprocess(tensor, orig):
    out = (tensor.clamp(0, 1).cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    result = Image.fromarray(out)
    return result.resize(orig, Image.LANCZOS) if result.size != orig else result

def _save_local(img, fname):
    """Save to local SSD only — fast. Drive sync happens once at the end."""
    img.save(CLEAN_IMG_LOCAL / fname)

def _copy_image(fpath):
    shutil.copy2(fpath, CLEAN_IMG_LOCAL / fpath.name)
    return fpath.name

needs_inpaint, needs_copy = [], []
for fpath in all_images:
    if fpath.name in inpaint_done:
        continue
    boxes = vision_cache.get(fpath.name)
    if boxes is None:
        continue
    (needs_inpaint if boxes else needs_copy).append(fpath)

print(f"\nStage 2 — Inpainting:")
print(f"  Needs inpainting : {len(needs_inpaint)}")
print(f"  Copy unchanged   : {len(needs_copy)}")

if needs_copy:
    with ThreadPoolExecutor(max_workers=SAVE_WORKERS) as pool:
        for name in tqdm(pool.map(_copy_image, needs_copy),
                         total=len(needs_copy), desc='Copying no-text images'):
            inpaint_done.add(name)
    INPAINT_LOG.write_text('\n'.join(inpaint_done))

size_groups = defaultdict(list)
for fpath in needs_inpaint:
    with open(fpath, 'rb') as fh:
        img = Image.open(fh)
        W, H = (img.width + 7) // 8 * 8, (img.height + 7) // 8 * 8
    size_groups[(W, H)].append(fpath)

total_batches = sum((len(g) + LAMA_BATCH_SIZE - 1) // LAMA_BATCH_SIZE
                    for g in size_groups.values())
print(f"  Size groups      : {len(size_groups)} | Total batches: {total_batches}")

n_inpainted   = 0
save_executor = ThreadPoolExecutor(max_workers=SAVE_WORKERS)

with torch.no_grad():
    pbar = tqdm(total=len(needs_inpaint), desc='Inpainting', unit='img')
    for (W, H), group_paths in size_groups.items():
        for i in range(0, len(group_paths), LAMA_BATCH_SIZE):
            batch = group_paths[i:i + LAMA_BATCH_SIZE]
            t_imgs, t_masks, origs, fnames = [], [], [], []
            for fpath in batch:
                with open(fpath, 'rb') as fh:
                    img = Image.open(fh).copy().convert('RGB')
                t_img, t_mask, orig = _preprocess(img, _make_mask(img.size, vision_cache[fpath.name]))
                t_imgs.append(t_img); t_masks.append(t_mask)
                origs.append(orig);   fnames.append(fpath.name)

            imgs_gpu  = torch.stack(t_imgs).to(lama_device)
            masks_gpu = torch.stack(t_masks).to(lama_device)
            outputs   = lama.model(imgs_gpu, masks_gpu)

            for fname, out, orig in zip(fnames, outputs, origs):
                save_executor.submit(_save_local, _postprocess(out, orig), fname)
                inpaint_done.add(fname)
                n_inpainted += 1

            pbar.update(len(batch))

            if n_inpainted % CHECKPOINT_EVERY == 0:
                INPAINT_LOG.write_text('\n'.join(inpaint_done))

    pbar.close()

save_executor.shutdown(wait=True)
INPAINT_LOG.write_text('\n'.join(inpaint_done))

print(f"\n✓ Inpainting done: {n_inpainted} images in {CLEAN_IMG_LOCAL}")
print("Syncing to Drive (one-time bulk copy)...")
shutil.copytree(CLEAN_IMG_LOCAL, CLEAN_IMG_DRIVE, dirs_exist_ok=True)
print(f"✓ Synced to Drive: {CLEAN_IMG_DRIVE}")

IMG_DIR = CLEAN_IMG_LOCAL
print(f"\n✓ IMG_DIR → {IMG_DIR}")

✓ Deps installed.
✓ Vision API client ready.
✓ LaMa ready on cuda:0.
Loaded Vision cache: 2165 images already processed.

Stage 1 — Vision API: 0 to annotate (2165 cached).

Inpainting log: 2065 images already done.

Stage 2 — Inpainting:
  Needs inpainting : 100
  Copy unchanged   : 0
  Size groups      : 42 | Total batches: 46


Inpainting: 100%|██████████| 100/100 [06:29<00:00,  3.89s/img]



✓ Inpainting done: 100 images in /content/rruff_clean_images
Syncing to Drive (one-time bulk copy)...
✓ Synced to Drive: /content/drive/My Drive/Classes/2026 Spring/advanced ml/rruff_processed/rruff_clean_images

✓ IMG_DIR → /content/rruff_clean_images


In [ ]:
#build ID → file mappings
ws = pd.read_csv(WS_CSV)
working_ids = set(ws['rruff_id'].str.upper())
print(f"Working set: {len(working_ids)} minerals")

# Map RRUFF ID → image file path (take first if multiple)
id_to_img = {}
for f in IMG_DIR.iterdir():
    if f.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
        continue
    rid, _ = parse_filename(f.name)
    if rid and rid in working_ids:
        if rid not in id_to_img:   # keep first image per mineral
            id_to_img[rid] = f

# Map RRUFF ID → list of Raman file paths
id_to_raman = defaultdict(list)
id_to_name  = {}
for f in RAMAN_DIR.iterdir():
    if f.suffix.lower() != '.txt':
        continue
    rid, mname = parse_filename(f.name)
    if rid and rid in working_ids:
        id_to_raman[rid].append(f)
        if mname and rid not in id_to_name:
            id_to_name[rid] = mname

# Minerals present in BOTH image and raman mappings
valid_ids = sorted(working_ids & set(id_to_img) & set(id_to_raman))
print(f"IDs with both image and Raman: {len(valid_ids)}")
print(f"Missing image : {len(working_ids - set(id_to_img))}")
print(f"Missing Raman : {len(working_ids - set(id_to_raman))}")

Working set: 1827 minerals
IDs with both image and Raman: 1827
Missing image : 0
Missing Raman : 0


In [ ]:
#selecting specimen files to work with per mineral species
from collections import defaultdict
import pandas as pd

ws = pd.read_csv(WS_CSV)
working_ids = set(ws['rruff_id'].str.upper())
print(f"Working set (RRUFF IDs): {len(working_ids)}")

def parse_wavelength(fname):
    """Extract laser wavelength (nm) from RRUFF Raman filename."""
    parts = Path(fname).stem.split('__')
    if len(parts) >= 4:
        try:
            wl = int(parts[3])
            if 100 <= wl <= 1200:
                return wl
        except ValueError:
            pass
    for part in parts:
        try:
            wl = int(part)
            if 100 <= wl <= 1200:
                return wl
        except ValueError:
            continue
    return None

def is_unoriented(fname):
    """Return True if Raman file is an unoriented measurement."""
    return 'unoriented' in Path(fname).stem.lower()

#building RRUFF ID -> mineral name mapping from Raman headers ─────────────────
#Raman filenames are used as the source for mineral names since they are more consistently named than the image files themselves. Sometimes the image files don't include the mineral name

id_to_name = {}
id_to_raman_all = defaultdict(list)   # all Raman files per RRUFF ID

for f in RAMAN_DIR.iterdir():
    if f.suffix.lower() != '.txt':
        continue
    rid, mname = parse_filename(f.name)
    if rid and rid in working_ids:
        id_to_raman_all[rid].append(f)
        if mname and rid not in id_to_name:
            id_to_name[rid] = mname

# one canonical 780/785nm Raman file per RRUFF ID
# Priority: 780nm unoriented > 780nm any > 785nm unoriented > 785nm any
TARGET_WLS = [780, 785]

def pick_raman_file(files):
    """Pick the best 780/785nm Raman file from a list."""
    candidates = {780: {'unoriented': [], 'any': []},
                  785: {'unoriented': [], 'any': []}}
    for f in files:
        wl = parse_wavelength(f.name)
        if wl in TARGET_WLS:
            key = 'unoriented' if is_unoriented(f.name) else 'any'
            candidates[wl][key].append(f)
    # Return in priority order
    for wl in [780, 785]:
        for key in ['unoriented', 'any']:
            if candidates[wl][key]:
                return candidates[wl][key][0], wl
    return None, None

id_to_raman_best = {}   # RRUFF ID -> single best Raman file path
id_to_raman_wl   = {}   # RRUFF ID -> wavelength used

for rid, files in id_to_raman_all.items():
    best_file, wl = pick_raman_file(files)
    if best_file:
        id_to_raman_best[rid] = best_file
        id_to_raman_wl[rid]   = wl

print(f"RRUFF IDs with 780/785nm Raman : {len(id_to_raman_best)}")
from collections import Counter
wl_used = Counter(id_to_raman_wl.values())
print(f"  780nm used : {wl_used.get(780, 0)}")
print(f"  785nm used : {wl_used.get(785, 0)}")

# building mineral name -> RRUFF ID mapping
'''
One node per unique mineral species name, for minerals with multiple RRUFF IDs ie multiple same species specimens:
prefer the specimen that has BOTH a 780/785nm Raman file AND an image.
in cases of ties, pick the RRUFF ID with the lowest R-number since that should be the oldest/most cited
'''

name_to_rids = defaultdict(list)
for rid in working_ids:
    name = id_to_name.get(rid)
    if name:
        name_to_rids[name].append(rid)

# build image lookup (all images, not filtered yet)
all_img_ids = set()
rid_to_img_path = {}
for f in IMG_DIR.iterdir():
    if f.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
        continue
    rid, _ = parse_filename(f.name)
    if rid and rid in working_ids:
        all_img_ids.add(rid)
        if rid not in rid_to_img_path:
            rid_to_img_path[rid] = f

def pick_canonical_rid(rids):
    """
    Pick one representative RRUFF ID for a mineral species.
    Prefer: has image AND 780/785nm Raman > has Raman only > has image only.
    Among ties: lowest R-number.
    """
    has_both   = [r for r in rids if r in rid_to_img_path and r in id_to_raman_best]
    has_raman  = [r for r in rids if r in id_to_raman_best and r not in rid_to_img_path]
    has_img    = [r for r in rids if r in rid_to_img_path and r not in id_to_raman_best]

    for pool in [has_both, has_raman, has_img, rids]:
        if pool:
            return sorted(pool)[0]   # lowest R-number
    return None

# Build final species-level mappings
name_to_canonical = {}   # mineral name -> RRUFF ID
for name, rids in name_to_rids.items():
    canonical = pick_canonical_rid(rids)
    if canonical:
        name_to_canonical[name] = canonical

#final working set: species with BOTH image and 780/785nm Raman
valid_names = sorted([
    name for name, rid in name_to_canonical.items()
    if rid in rid_to_img_path and rid in id_to_raman_best
])
valid_rids  = [name_to_canonical[name] for name in valid_names]

# convenience lookups for downstream cells
id_to_img   = {rid: rid_to_img_path[rid] for rid in valid_rids}
id_to_raman = {rid: id_to_raman_best[rid] for rid in valid_rids}   # single file

print(f"\nFinal working set:")
print(f"  Unique mineral species with image + 780/785nm Raman : {len(valid_names)}")
print(f"  (dropped {len(working_ids) - len(valid_names)} RRUFF IDs — "
      f"no 780/785nm file or no image)")
print(f"\nSample species -> canonical RRUFF ID:")
for name, rid in list(name_to_canonical.items())[:5]:
    wl = id_to_raman_wl.get(rid, '?')
    print(f"  {name:<35} {rid}  ({wl}nm)")

Working set (RRUFF IDs): 1827
RRUFF IDs with 780/785nm Raman : 1735
  780nm used : 1241
  785nm used : 494

Final working set:
  Unique mineral species with image + 780/785nm Raman : 1264
  (dropped 563 RRUFF IDs — no 780/785nm file or no image)

Sample species -> canonical RRUFF ID:
  Lamprophyllite                      R070284  (785nm)
  Bastnasite-(Ce)                     R060550  (780nm)
  Metakottigite                       R130695  (780nm)
  Wupatkiite                          R060515  (780nm)
  Manandonite                         R060968  (785nm)


In [ ]:
#CLIP image embeddings (one image per mineral species)

CLIP_MODEL_NAME = 'ViT-B-32'
CLIP_PRETRAINED = 'openai'
BATCH_SIZE      = 256

CLIP_EMB_PATH = Path("/content/drive/My Drive/Classes/2026 Spring/advanced ml/rruff_processed/clip_embeddings_v2.pt")
CLIP_EMB_PATH.parent.mkdir(parents=True, exist_ok=True)

if CLIP_EMB_PATH.exists():
    print("Loading cached CLIP embeddings...")
    clip_embs = torch.load(CLIP_EMB_PATH, weights_only=False)
    print(f"Loaded {len(clip_embs)} cached embeddings.")
    remaining_rids = [rid for rid in valid_rids if rid not in clip_embs]
    print(f"Remaining to process: {len(remaining_rids)}")
else:
    clip_embs     = {}
    remaining_rids = valid_rids

if remaining_rids:
    print(f"Computing CLIP embeddings for {len(remaining_rids)} minerals...")
    clip_model, _, preprocess = open_clip.create_model_and_transforms(
        CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED
    )
    clip_model = clip_model.to(DEVICE).eval()

    batches = [remaining_rids[i:i+BATCH_SIZE]
               for i in range(0, len(remaining_rids), BATCH_SIZE)]

    SAVE_EVERY = 5
    with torch.no_grad():
        for batch_num, batch_rids in enumerate(tqdm(batches, desc='CLIP')):
            imgs, ok_rids = [], []
            for rid in batch_rids:
                try:
                    img = Image.open(id_to_img[rid]).convert('RGB')
                    imgs.append(preprocess(img))
                    ok_rids.append(rid)
                except Exception as e:
                    print(f"  Warning: could not load {id_to_img[rid].name}: {e}")
            if not imgs:
                continue
            batch_tensor = torch.stack(imgs).to(DEVICE)
            embs = clip_model.encode_image(batch_tensor)
            embs = F.normalize(embs, dim=-1).cpu()
            for rid, emb in zip(ok_rids, embs):
                clip_embs[rid] = emb
            if (batch_num + 1) % SAVE_EVERY == 0:
                torch.save(clip_embs, CLIP_EMB_PATH)

    torch.save(clip_embs, CLIP_EMB_PATH)
    print("Saved CLIP embeddings to Drive.")
    del clip_model   # free GPU memory

print(f"CLIP embeddings: {len(clip_embs)} minerals, dim={next(iter(clip_embs.values())).shape[0]}")

Computing CLIP embeddings for 1264 minerals...


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIP: 100%|██████████| 5/5 [00:20<00:00,  4.07s/it]

Saved CLIP embeddings to Drive.
CLIP embeddings: 1264 minerals, dim=512


In [ ]:
#raman embeddings are made from the mineral species' 780/785nm file

from scipy.interpolate import interp1d
from sklearn.decomposition import PCA
import joblib

RAMAN_GRID = np.linspace(100, 4000, 1000)
RAMAN_DIM  = 256
RAMAN_EMB_PATH = OUT_DIR / 'raman_embeddings_v2.pt'

def parse_raman_file(fpath):
    """Parse RRUFF Raman txt -> (wavenumbers, intensities)."""
    wn, intensity = [], []
    with open(fpath, 'r', errors='ignore') as fh:
        for line in fh:
            line = line.strip()
            if line.startswith('#') or not line:
                continue
            parts = line.replace(',', ' ').split()
            if len(parts) >= 2:
                try:
                    wn.append(float(parts[0]))
                    intensity.append(float(parts[1]))
                except ValueError:
                    continue
    return np.array(wn), np.array(intensity)

def spectrum_to_grid(wn, intensity, grid=RAMAN_GRID):
    """Interpolate spectrum onto common grid and normalize to [0,1]."""
    min_len = min(len(wn), len(intensity))
    wn, intensity = wn[:min_len], intensity[:min_len]
    if len(wn) < 4:
        return None
    mask = (wn >= grid[0]) & (wn <= grid[-1])
    wn, intensity = wn[mask], intensity[mask]
    if len(wn) < 4:
        return None
    sort_idx = np.argsort(wn)
    wn, intensity = wn[sort_idx], intensity[sort_idx]
    mn, mx = intensity.min(), intensity.max()
    if mx - mn < 1e-9:
        return None
    intensity = (intensity - mn) / (mx - mn)
    f = interp1d(wn, intensity, kind='linear', bounds_error=False, fill_value=0.0)
    return f(grid).astype(np.float32)

print("Processing Raman spectra (single 780/785nm file per mineral)...")
raman_spectra_raw = {}
failed_raman = []

for rid in tqdm(valid_rids, desc='Raman'):
    fpath = id_to_raman[rid]   # single file, no averaging needed
    wn, intensity = parse_raman_file(fpath)
    gridded = spectrum_to_grid(wn, intensity)
    if gridded is not None:
        raman_spectra_raw[rid] = gridded
    else:
        failed_raman.append(rid)

print(f"Successfully processed : {len(raman_spectra_raw)} minerals")
print(f"Failed (bad spectrum)  : {len(failed_raman)}")
if failed_raman:
    print(f"  Examples: {failed_raman[:5]}")

# ── PCA compression ───────────────────────────────────────────────────────────
# Only keep minerals that have both CLIP and Raman
raman_ids = [rid for rid in valid_rids
             if rid in raman_spectra_raw and rid in clip_embs]

raman_matrix = np.stack([raman_spectra_raw[rid] for rid in raman_ids])
print(f"\nRaman matrix: {raman_matrix.shape}")

pca = PCA(n_components=RAMAN_DIM, random_state=42)
raman_compressed = pca.fit_transform(raman_matrix)
print(f"Explained variance ({RAMAN_DIM} components): {pca.explained_variance_ratio_.sum():.3f}")

joblib.dump(pca, OUT_DIR / 'raman_pca_v2.pkl')

raman_embs = {
    rid: torch.tensor(raman_compressed[i], dtype=torch.float32)
    for i, rid in enumerate(raman_ids)
}
torch.save(raman_embs, RAMAN_EMB_PATH)
print(f"Raman embeddings saved: {len(raman_embs)} minerals, dim={RAMAN_DIM}")

# ── Update valid_names and valid_rids to only those with both embeddings ───────
valid_rids_final  = [rid for rid in valid_rids
                     if rid in clip_embs and rid in raman_embs]
valid_names_final = [name for name, rid in zip(valid_names, valid_rids)
                     if rid in clip_embs and rid in raman_embs]

print(f"\nFinal node count (image + 780/785nm Raman + both embeddings): {len(valid_rids_final)}")
print(f"  (dropped {len(valid_rids) - len(valid_rids_final)} due to bad spectra or image load failure)")


Processing Raman spectra (single 780/785nm file per mineral)...


Raman: 100%|██████████| 1264/1264 [00:02<00:00, 427.84it/s]


Successfully processed : 1264 minerals
Failed (bad spectrum)  : 0

Raman matrix: (1264, 1000)
Explained variance (256 components): 1.000
Raman embeddings saved: 1264 minerals, dim=256

Final node count (image + 780/785nm Raman + both embeddings): 1264
  (dropped 0 due to bad spectra or image load failure)


In [ ]:
#PCA compression of Raman spectra
from sklearn.decomposition import PCA
import joblib

PCA_PATH = OUT_DIR / 'raman_pca.pkl'

# build matrix of valid spectra
raman_ids   = [rid for rid in valid_ids if rid in raman_spectra_raw and rid in clip_embs]
raman_matrix = np.stack([raman_spectra_raw[rid] for rid in raman_ids])  #(N, 1000)
print(f"Raman matrix shape: {raman_matrix.shape}")

# fit PCA
pca = PCA(n_components=RAMAN_DIM, random_state=42)
raman_compressed = pca.fit_transform(raman_matrix)  #(N, 256)
print(f"Explained variance (256 components): {pca.explained_variance_ratio_.sum():.3f}")

joblib.dump(pca, PCA_PATH)
print(f"PCA model saved → {PCA_PATH}")

# build dict: rruff_id -> raman embedding tensor
raman_embs = {
    rid: torch.tensor(raman_compressed[i], dtype=torch.float32)
    for i, rid in enumerate(raman_ids)
}
torch.save(raman_embs, OUT_DIR / 'raman_embeddings.pt')
print(f"Raman embeddings saved: {len(raman_embs)} minerals, dim=256")

Raman matrix shape: (1264, 1000)
Explained variance (256 components): 1.000
PCA model saved → /content/rruff_processed/raman_pca.pkl
Raman embeddings saved: 1264 minerals, dim=256


In [ ]:
#building node list & feature matrix
#final node set = minerals with both CLIP and Raman embeddings

final_ids = sorted(set(clip_embs) & set(raman_embs))
print(f"Final node count: {len(final_ids)}")

#build mineral name list using Raman filename name, fall back to RRUFF ID
mineral_names = [id_to_name.get(rid, rid) for rid in final_ids]

#encode mineral names as integer labels for classification
le = LabelEncoder()
labels = le.fit_transform(mineral_names)  # int per unique mineral name
n_classes = len(le.classes_)
print(f"Unique mineral classes (names): {n_classes}")

#concatenate CLIP + Raman to node feature matrix
clip_dim  = next(iter(clip_embs.values())).shape[0]   # 512
raman_dim = next(iter(raman_embs.values())).shape[0]  # 256
feat_dim  = clip_dim + raman_dim
print(f"Node feature dim: {clip_dim} (CLIP) + {raman_dim} (Raman) = {feat_dim}")

X = torch.zeros(len(final_ids), feat_dim)
for i, rid in enumerate(final_ids):
    X[i, :clip_dim]  = clip_embs[rid]
    X[i, clip_dim:]  = raman_embs[rid]

y = torch.tensor(labels, dtype=torch.long)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}, n_classes: {n_classes}")

Final node count: 1264
Unique mineral classes (names): 1264
Node feature dim: 512 (CLIP) + 256 (Raman) = 768
X shape: torch.Size([1264, 768])
y shape: torch.Size([1264]), n_classes: 1264


In [ ]:
import re
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

CRYSTAL_RE  = re.compile(r'crystal system:\s*(\w+)', re.IGNORECASE)
LOCALITY_RE = re.compile(r'##LOCALITY=(.+)')
CHEM_RE     = re.compile(r'##IDEAL CHEMISTRY=(.+)')
DESC_RE     = re.compile(r'##DESCRIPTION=(.+)')
ELEMENT_RE  = re.compile(r'([A-Z][a-z]?)')

VALID_ELEMENTS = {
    'H','He','Li','Be','B','C','N','O','F','Ne','Na','Mg','Al','Si','P',
    'S','Cl','Ar','K','Ca','Sc','Ti','V','Cr','Mn','Fe','Co','Ni','Cu',
    'Zn','Ga','Ge','As','Se','Br','Kr','Rb','Sr','Y','Zr','Nb','Mo',
    'Tc','Ru','Rh','Pd','Ag','Cd','In','Sn','Sb','Te','I','Xe','Cs',
    'Ba','La','Ce','Pr','Nd','Pm','Sm','Eu','Gd','Tb','Dy','Ho','Er',
    'Tm','Yb','Lu','Hf','Ta','W','Re','Os','Ir','Pt','Au','Hg','Tl',
    'Pb','Bi','Po','At','Rn','Fr','Ra','Ac','Th','Pa','U'
}

def parse_raman_metadata(fpath):
    crystal, locality, elements, description = None, None, set(), None
    with open(fpath, 'r', errors='ignore') as fh:
        for line in fh:
            if line.startswith('##IDEAL CHEMISTRY='):
                formula = CHEM_RE.match(line.strip())
                if formula:
                    raw_elems = ELEMENT_RE.findall(formula.group(1))
                    elements = {e for e in raw_elems if e in VALID_ELEMENTS}
            elif line.startswith('##CELL PARAMETERS='):
                cm = CRYSTAL_RE.search(line)
                if cm:
                    crystal = cm.group(1).lower()
            elif line.startswith('##LOCALITY='):
                lm = LOCALITY_RE.match(line.strip())
                if lm:
                    locality = lm.group(1).strip().lower()
            elif line.startswith('##DESCRIPTION='):
                dm = DESC_RE.match(line.strip())
                if dm:
                    description = dm.group(1).strip()
            elif not line.startswith('#'):
                break
    return crystal, locality, elements, description

def process_metadata(mineral_name):
    crystal_sys, locality, elements, descriptions = None, None, set(), []
    # Get the canonical RRUFF ID for the mineral name
    rid = name_to_canonical.get(mineral_name)
    if not rid:
        return mineral_name, {
            'crystal': None, 'locality': None, 'elements': set(), 'description': None
        }

    # id_to_raman[rid] now holds a single PosixPath object, not a list
    fpath = id_to_raman[rid]
    c, l, e, d = parse_raman_metadata(fpath)
    if c: crystal_sys = c
    if l: locality    = l
    elements |= e
    if d and d not in descriptions:
        descriptions.append(d)
    return mineral_name, {
        'crystal':     crystal_sys,
        'locality':    locality,
        'elements':    elements,
        'description': ' '.join(descriptions) if descriptions else None,
    }

print("Parsing Raman metadata (parallelised across 8 threads)...")
metadata = {}

with ThreadPoolExecutor(max_workers=8) as executor:
    for mineral_name, meta in tqdm(
        executor.map(process_metadata, valid_names_final),
        total=len(valid_names_final), desc='Metadata'
    ):
        metadata[mineral_name] = meta

has_crystal     = sum(1 for v in metadata.values() if v['crystal'])
has_locality    = sum(1 for v in metadata.values() if v['locality'])
has_elements    = sum(1 for v in metadata.values() if v['elements'])
has_description = sum(1 for v in metadata.values() if v['description'])
print(f"Minerals with crystal system : {has_crystal}")
print(f"Minerals with locality       : {has_locality}")
print(f"Minerals with elements       : {has_elements}")
print(f"Minerals with description    : {has_description}")

Parsing Raman metadata (parallelised across 8 threads)...


Metadata: 100%|██████████| 1264/1264 [00:00<00:00, 7427.64it/s]

Minerals with crystal system : 1117
Minerals with locality       : 1264
Minerals with elements       : 1252
Minerals with description    : 1264


In [ ]:
# ── Cell 9: Compute edges ───────────────────────────────
# Build index map: rruff_id -> node index
id_to_idx = {rid: i for i, rid in enumerate(final_ids)}
N = len(final_ids)

edges_chem    = []  # type 0 — shared elements
edges_crystal = []  # type 1 — shared crystal system
edges_locality= []  # type 2 — same locality

# Group by crystal system
crystal_groups = defaultdict(list)
for mineral_name, meta in metadata.items():
    if meta['crystal']:
        rruff_id = name_to_canonical.get(mineral_name)
        if rruff_id:
            crystal_groups[meta['crystal']].append(id_to_idx[rruff_id])

# Group by locality
locality_groups = defaultdict(list)
for mineral_name, meta in metadata.items():
    if meta['locality']:
        rruff_id = name_to_canonical.get(mineral_name)
        if rruff_id:
            locality_groups[meta['locality']].append(id_to_idx[rruff_id])

print(f"Unique crystal systems : {len(crystal_groups)}")
print(f"Unique localities      : {len(locality_groups)}")

# Crystal system edges — connect all minerals in same system
# Cap group size at 200 to avoid O(n^2) explosion in huge groups
MAX_GROUP = 200
for group in tqdm(crystal_groups.values(), desc='Crystal edges'):
    if len(group) > MAX_GROUP:
        group = group[:MAX_GROUP]
    for i in range(len(group)):
        for j in range(i+1, len(group)):
            edges_crystal.append((group[i], group[j]))
            edges_crystal.append((group[j], group[i]))

# Locality edges
for group in tqdm(locality_groups.values(), desc='Locality edges'):
    if len(group) > MAX_GROUP:
        group = group[:MAX_GROUP]
    for i in range(len(group)):
        for j in range(i+1, len(group)):
            edges_locality.append((group[i], group[j]))
            edges_locality.append((group[j], group[i]))

# Element overlap edges
element_groups = defaultdict(list)
for mineral_name, meta in metadata.items():
    rruff_id = name_to_canonical.get(mineral_name)
    if rruff_id:
        for elem in meta['elements']:
            element_groups[elem].append(id_to_idx[rruff_id])

# Only connect pairs sharing >= 2 elements (reduces noise from ubiquitous O, Si)
print("Computing element overlap edges (sharing >= 2 elements)...")
from itertools import combinations
pair_shared = defaultdict(int)
for elem, group in element_groups.items():
    if len(group) > MAX_GROUP:
        group = group[:MAX_GROUP]
    for a, b in combinations(group, 2):
        pair_shared[(min(a,b), max(a,b))] += 1

for (a, b), count in pair_shared.items():
    if count >= 2:
        edges_chem.append((a, b))
        edges_chem.append((b, a))

print(f"\nEdge counts:")
print(f"  Chemical (>=2 shared elements) : {len(edges_chem)//2:6d} undirected pairs")
print(f"  Crystal system                 : {len(edges_crystal)//2:6d} undirected pairs")
print(f"  Locality co-occurrence         : {len(edges_locality)//2:6d} undirected pairs")

Unique crystal systems : 6
Unique localities      : 1048


Locality edges: 100%|██████████| 1048/1048 [00:00<00:00, 805060.55it/s]

Computing element overlap edges (sharing >= 2 elements)...



Edge counts:
  Chemical (>=2 shared elements) :  47657 undirected pairs
  Crystal system                 :  72874 undirected pairs
  Locality co-occurrence         :    703 undirected pairs


In [ ]:
# ── Cell 10: Assemble PyG Data object ────────────────────────────────────────

def edges_to_tensor(edge_list):
    if not edge_list:
        return torch.zeros(2, 0, dtype=torch.long)
    return torch.tensor(edge_list, dtype=torch.long).t().contiguous()

ei_chem     = edges_to_tensor(edges_chem)
ei_crystal  = edges_to_tensor(edges_crystal)
ei_locality = edges_to_tensor(edges_locality)

# Concatenate all edges, with type labels
edge_index = torch.cat([ei_chem, ei_crystal, ei_locality], dim=1)
edge_type  = torch.cat([
    torch.zeros(ei_chem.size(1),     dtype=torch.long),
    torch.ones(ei_crystal.size(1),   dtype=torch.long),
    torch.full((ei_locality.size(1),), 2, dtype=torch.long),
])

# Remove duplicate edges (keep edge_type of first occurrence)
# Sort by (src, dst) and deduplicate
edge_df = pd.DataFrame({
    'src':  edge_index[0].numpy(),
    'dst':  edge_index[1].numpy(),
    'type': edge_type.numpy(),
})
edge_df = edge_df.drop_duplicates(subset=['src','dst']).reset_index(drop=True)
edge_index = torch.tensor(edge_df[['src','dst']].values.T, dtype=torch.long)
edge_type  = torch.tensor(edge_df['type'].values,          dtype=torch.long)

print(f"Total edges (after dedup): {edge_index.size(1)}")
print(f"Edge type distribution:")
for t, name in [(0,'Chemical'), (1,'Crystal'), (2,'Locality')]:
    print(f"  Type {t} ({name:10s}): {(edge_type==t).sum().item()}")

# Build Data object
data = Data(
    x           = X,
    edge_index  = edge_index,
    edge_type   = edge_type,
    y           = y,
    num_classes = n_classes,
)

# Store metadata as plain Python lists (not in Data to avoid issues)
print(f"\nData object:")
print(data)
print(f"Nodes          : {data.num_nodes}")
print(f"Edges          : {data.num_edges}")
print(f"Feature dim    : {data.num_features}")
print(f"Classes        : {data.num_classes}")

Total edges (after dedup): 227672
Edge type distribution:
  Type 0 (Chemical  ): 95314
  Type 1 (Crystal   ): 131378
  Type 2 (Locality  ): 980

Data object:
Data(x=[1264, 768], edge_index=[2, 227672], y=[1264], edge_type=[227672], num_classes=1264)
Nodes          : 1264
Edges          : 227672
Feature dim    : 768
Classes        : 1264


In [ ]:
# ── Cell 11: Train/val/test split ────────────────────────────────────────────
from sklearn.model_selection import train_test_split

N = data.num_nodes
indices = np.arange(N)

# 80/10/10 split, stratified by class label where possible
try:
    train_idx, temp_idx = train_test_split(
        indices, test_size=0.2, random_state=42, stratify=y.numpy()
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5, random_state=42, stratify=y.numpy()[temp_idx]
    )
except ValueError:
    # Stratify fails if some classes have <2 samples — fall back to random
    print("Stratified split failed, using random split.")
    train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42)
    val_idx, test_idx   = train_test_split(temp_idx, test_size=0.5, random_state=42)

train_mask = torch.zeros(N, dtype=torch.bool)
val_mask   = torch.zeros(N, dtype=torch.bool)
test_mask  = torch.zeros(N, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx]     = True
test_mask[test_idx]   = True

data.train_mask = train_mask
data.val_mask   = val_mask
data.test_mask  = test_mask

print(f"Train: {train_mask.sum().item()} | Val: {val_mask.sum().item()} | Test: {test_mask.sum().item()}")

Stratified split failed, using random split.
Train: 1011 | Val: 126 | Test: 127


In [ ]:
import pickle, shutil

DATA_PATH = OUT_DIR / 'mineral_graph.pt'
META_PATH = OUT_DIR / 'mineral_meta.pkl'

torch.save(data, DATA_PATH)

meta = {
    'final_ids':     final_ids,
    'mineral_names': mineral_names,
    'label_encoder': le,
    'n_classes':     n_classes,
    'clip_dim':      clip_dim,
    'raman_dim':     raman_dim,
    'feat_dim':      feat_dim,
}
with open(META_PATH, 'wb') as f:
    pickle.dump(meta, f)

print(f"Saved graph data → {DATA_PATH}")
print(f"Saved metadata   → {META_PATH}")

# ── Sync all outputs to Drive for persistence ─────────────────────────────────
print("\nSyncing outputs to Drive...")
shutil.copytree(str(OUT_DIR), str(DRIVE_OUT), dirs_exist_ok=True)
print(f"Outputs synced → {DRIVE_OUT}")

print()
print("=" * 50)
print("PREPROCESSING COMPLETE")
print("=" * 50)
print(f"  Nodes      : {data.num_nodes}")
print(f"  Edges      : {data.num_edges}")
print(f"  Features   : {data.num_features}  (CLIP {clip_dim} + Raman {raman_dim})")
print(f"  Classes    : {n_classes}")
print(f"  Train/Val/Test: {train_mask.sum()}/{val_mask.sum()}/{test_mask.sum()}")
print()
print("Next step: load mineral_graph.pt and train GNN.")


Saved graph data → /content/rruff_processed/mineral_graph.pt
Saved metadata   → /content/rruff_processed/mineral_meta.pkl

Syncing outputs to Drive...
Outputs synced → /content/drive/My Drive/Classes/2026 Spring/advanced ml/rruff_processed

PREPROCESSING COMPLETE
  Nodes      : 1264
  Edges      : 227672
  Features   : 768  (CLIP 512 + Raman 256)
  Classes    : 1264
  Train/Val/Test: 1011/126/127

Next step: load mineral_graph.pt and train GNN.


In [ ]:
expected = [
    'mineral_graph.pt', 'mineral_meta.pkl', 'clip_embeddings_v2.pt',
    'raman_embeddings_v2.pt', 'raman_pca_v2.pkl',
    'vision_cache.json', 'inpaint_done.txt'
]
for fname in expected:
    p = DRIVE_OUT / fname
    status = "✓" if p.exists() else "✗ MISSING"
    print(f"  {status}  {fname}")

clean_count = len(list((DRIVE_OUT / 'rruff_clean_images').iterdir()))
print(f" rruff_clean_images/ ({clean_count} files)")

  ✓  mineral_graph.pt
  ✓  mineral_meta.pkl
  ✓  clip_embeddings_v2.pt
  ✓  raman_embeddings_v2.pt
  ✓  raman_pca_v2.pkl
  ✓  vision_cache.json
  ✓  inpaint_done.txt
  ✓  rruff_clean_images/ (2165 files)
